In [0]:
%sql

CREATE OR REPLACE TABLE retail_medallion_ws.default.silver_erp_customer_dedup AS
SELECT CID, BDATE, GEN
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY SUBSTRING(CID, 4, LEN(CID))
            ORDER BY BDATE DESC NULLS LAST
        ) AS rn
    FROM retail_medallion_ws.default.silver_erp_customer
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql

CREATE OR REPLACE TABLE retail_medallion_ws.default.silver_erp_location_dedup AS
SELECT CID, CNTRY
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY CID
            ORDER BY CNTRY DESC NULLS LAST
        ) AS rn
    FROM retail_medallion_ws.default.silver_erp_location
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql

CREATE OR REPLACE TABLE retail_medallion_ws.default.dim_customer AS
SELECT
    ROW_NUMBER() OVER (ORDER BY c.cst_id) AS customer_sk,
    c.cst_id AS customer_id,
    c.cst_key AS customer_key,
    c.cst_firstname AS first_name,
    c.cst_lastname AS last_name,
    CASE
        WHEN c.cst_marital_status = 'M' THEN 'Married'
        WHEN c.cst_marital_status = 'S' THEN 'Single'
        ELSE 'n/a'
    END AS marital_status,
    CASE
        WHEN UPPER(c.cst_gndr) = 'M' OR UPPER(e.GEN) = 'MALE'   THEN 'Male'
        WHEN UPPER(c.cst_gndr) = 'F' OR UPPER(e.GEN) = 'FEMALE' THEN 'Female'
        ELSE 'n/a'
    END AS gender,
    e.BDATE AS birth_date,
    COALESCE(l.CNTRY, 'n/a') AS country
FROM retail_medallion_ws.default.silver_customer c
LEFT JOIN retail_medallion_ws.default.silver_erp_customer_dedup e
    ON c.cst_key = SUBSTRING(e.CID, 4, LEN(e.CID))
LEFT JOIN retail_medallion_ws.default.silver_erp_location_dedup l
    ON c.cst_key = REPLACE(l.CID, '-', '');

num_affected_rows,num_inserted_rows


In [0]:
SELECT customer_sk, customer_id, gender, country, birth_date
FROM retail_medallion_ws.default.dim_customer
LIMIT 10;

customer_sk,customer_id,gender,country,birth_date
1,11000,Male,Australia,1971-10-06
2,11001,Male,Australia,1976-05-10
3,11002,Male,Australia,1971-02-09
4,11003,Female,Australia,1973-08-14
5,11004,Female,Australia,1979-08-05
6,11005,Male,Australia,1976-08-01
7,11006,Female,Australia,1976-12-02
8,11007,Male,Australia,1969-11-06
9,11008,Female,Australia,1975-07-04
10,11009,Male,Australia,1969-09-29


In [0]:
SELECT customer_id, COUNT(*) AS cnt
FROM retail_medallion_ws.default.dim_customer
GROUP BY customer_id
HAVING COUNT(*) > 1;

customer_id,cnt


In [0]:
%sql

CREATE OR REPLACE TABLE retail_medallion_ws.default.dim_product AS
SELECT * FROM (
    SELECT
        ROW_NUMBER() OVER (ORDER BY p.prd_id) AS product_sk,
        p.prd_id AS product_id,
        SUBSTRING(p.prd_key, 7, LEN(p.prd_key)) AS product_key,
        p.prd_nm AS product_name,
        p.prd_cost AS product_cost,
        p.prd_line AS product_line,
        cat.CAT AS category,
        cat.SUBCAT AS subcategory,
        ROW_NUMBER() OVER (
            PARTITION BY SUBSTRING(p.prd_key, 7, LEN(p.prd_key))
            ORDER BY p.prd_start_dt DESC
        ) AS rn
    FROM retail_medallion_ws.default.silver_product_info p
    LEFT JOIN retail_medallion_ws.default.silver_erp_category cat
        ON REPLACE(SUBSTRING(p.prd_key, 1, 5), '-', '_') = cat.ID
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM retail_medallion_ws.default.dim_product LIMIT 10;

product_sk,product_id,product_key,product_name,product_cost,product_line,category,subcategory,rn
1,210,FR-R92B-58,HL Road Frame - Black- 58,null,R,Components,Road Frames,1
2,211,FR-R92R-58,HL Road Frame - Red- 58,null,R,Components,Road Frames,1
5,214,HL-U509-R,Sport-100 Helmet- Red,13,S,Accessories,Helmets,1
8,217,HL-U509,Sport-100 Helmet- Black,13,S,Accessories,Helmets,1
9,218,SO-B909-M,Mountain Bike Socks- M,3,M,Clothing,Socks,1
10,219,SO-B909-L,Mountain Bike Socks- L,3,M,Clothing,Socks,1
13,222,HL-U509-B,Sport-100 Helmet- Blue,13,S,Accessories,Helmets,1
16,225,CA-1098,AWC Logo Cap,7,S,Clothing,Caps,1
19,228,LJ-0192-S,Long-Sleeve Logo Jersey- S,38,S,Clothing,Jerseys,1
22,231,LJ-0192-M,Long-Sleeve Logo Jersey- M,38,S,Clothing,Jerseys,1


In [0]:
%sql

CREATE OR REPLACE TABLE retail_medallion_ws.default.dim_date AS
WITH date_range AS (
    SELECT
        MIN(try_to_date(CAST(sls_order_dt AS STRING), 'yyyyMMdd')) AS min_date,
        MAX(try_to_date(CAST(sls_order_dt AS STRING), 'yyyyMMdd')) AS max_date
    FROM retail_medallion_ws.default.silver_sales
    WHERE sls_order_dt != 0
)
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_sk,
    d AS full_date,
    YEAR(d) AS year,
    QUARTER(d) AS quarter,
    MONTH(d) AS month,
    date_format(d, 'MMMM') AS month_name,
    DAY(d) AS day,
    date_format(d, 'EEEE') AS day_name,
    CASE WHEN DAYOFWEEK(d) IN (1, 7) THEN true ELSE false END AS is_weekend
FROM date_range
LATERAL VIEW explode(sequence(min_date, max_date, interval 1 day)) AS d;

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) AS total_dates, MIN(full_date) AS earliest, MAX(full_date) AS latest
FROM retail_medallion_ws.default.dim_date;

total_dates,earliest,latest
1127,2010-12-29,2014-01-28


In [0]:
%sql

CREATE OR REPLACE TABLE retail_medallion_ws.default.fact_sales AS
SELECT
    s.sls_ord_num AS order_id,
    p.product_sk,
    c.customer_sk,
    d.date_sk AS order_date_sk,
    s.sls_order_dt AS order_date,
    s.sls_quantity AS quantity,
    ABS(s.sls_price) AS unit_price,
    CASE
        WHEN s.sls_sales IS NULL
             OR s.sls_sales <= 0
             OR s.sls_sales <> s.sls_quantity * ABS(s.sls_price)
        THEN s.sls_quantity * ABS(s.sls_price)
        ELSE s.sls_sales
    END AS sales_amount
FROM retail_medallion_ws.default.silver_sales s
LEFT JOIN retail_medallion_ws.default.dim_product p
    ON s.sls_prd_key = p.product_key
LEFT JOIN retail_medallion_ws.default.dim_customer c
    ON s.sls_cust_id = c.customer_id
LEFT JOIN retail_medallion_ws.default.dim_date d
    ON try_to_date(CAST(s.sls_order_dt AS STRING), 'yyyyMMdd') = d.full_date;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM retail_medallion_ws.default.fact_sales LIMIT 10;

order_id,product_sk,customer_sk,order_date_sk,order_date,quantity,unit_price,sales_amount
SO43697,101,10769,20101229,20101229,1,3578,3578
SO43698,137,17390,20101229,20101229,1,3400,3400
SO43699,137,14864,20101229,20101229,1,3400,3400
SO43700,128,3502,20101229,20101229,1,699,699
SO43701,137,4,20101229,20101229,1,3400,3400
SO43702,102,16646,20101230,20101230,1,3578,3578
SO43703,101,5625,20101230,20101230,1,3578,3578
SO43704,142,6,20101230,20101230,1,3375,3375
SO43705,135,12,20101230,20101230,1,3400,3400
SO43706,103,16622,20101231,20101231,1,3578,3578


In [0]:
SELECT 'dim_customer' AS table_name, COUNT(*) AS records FROM retail_medallion_ws.default.dim_customer
UNION ALL
SELECT 'dim_product', COUNT(*) FROM retail_medallion_ws.default.dim_product
UNION ALL
SELECT 'dim_date', COUNT(*) FROM retail_medallion_ws.default.dim_date
UNION ALL
SELECT 'fact_sales', COUNT(*) FROM retail_medallion_ws.default.fact_sales;

table_name,records
dim_customer,18484
dim_product,295
dim_date,1127
fact_sales,60383


In [0]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN product_sk IS NULL THEN 1 ELSE 0 END) AS unmatched_product,
    SUM(CASE WHEN customer_sk IS NULL THEN 1 ELSE 0 END) AS unmatched_customer,
    SUM(CASE WHEN order_date_sk IS NULL THEN 1 ELSE 0 END) AS unmatched_date
FROM retail_medallion_ws.default.fact_sales;

total_rows,unmatched_product,unmatched_customer,unmatched_date
60383,0,0,19
